# Supplemental Tables for PepBind3D Manuscript

Generates four supplementary tables:

| Table | Content | Source |
|-------|---------|--------|
| **S1** | Deduplication decision logic | Hand-encoded from `clean_peplist()` in `IEDBTestPipeline.py` |
| **S2** | Per-allele Spearman correlations (IC50 and Kd) | Computed from `metadata.csv` |
| **S3** | Structural validation pairs | Loaded from `rmsd_per_pair.csv` |
| **S4** | Per-allele dataset composition | Derived from `metadata.csv` |

## Imports and paths

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from pathlib import Path

METADATA_CSV  = Path('/home/huntek1/main_project/data/IEDB_data_clean/metadata.csv')
RMSD_CSV      = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/01_structural_regen/rmsd_per_pair.csv')
OUT_DIR       = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/supplemental_tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

pd.set_option('display.max_colwidth', None)  # show full text, no truncation

## Table S1 — Deduplication decision logic

Decision rules applied to duplicate IEDB entries for the same (epitope, assay-type) combination, 
reflecting the logic in `clean_peplist()` of `IEDBTestPipeline.py`.

In [21]:
s1_rows = [
    {'Step': 'Pre-filter', 'Case': 'Missing quantitative measurement or assay units',
     'Condition': '`Quantitative_Measurement` or `Assay_Units` is NaN',
     'Action': 'Drop record'},
    {'Step': 'Pre-filter', 'Case': 'Variant Kd assay labels',
     'Condition': '`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}',
     'Action': 'Normalize to dissociation constant (KD) prior to filtering'},
    {'Step': 'Pre-filter', 'Case': 'Non-binding-affinity assay',
     'Condition': '`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}',
     'Action': 'Drop record'},

    {'Step': 'Dedup (n=1)', 'Case': 'Single record',
     'Condition': '—', 'Action': 'Keep as-is'},

    {'Step': 'Dedup (n=2)', 'Case': 'PubMed provenance asymmetric',
     'Condition': 'One record has a PubMed ID, the other does not',
     'Action': 'Drop the record without PubMed ID'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values agree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| ≤ 10 nM',
     'Action': 'Keep one representative (the first)'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values disagree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| > 10 nM',
     'Action': 'Drop both records'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values agree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| < 10 nM',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values disagree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| ≥ 10 nM',
     'Action': 'Flag for manual review against source publications'},

    {'Step': 'Dedup (n>2)', 'Case': 'Mixed PubMed status',
     'Condition': 'Some records have PubMed ID, others do not',
     'Action': 'Drop records without PubMed ID first, then re-evaluate'},
    {'Step': 'Dedup (n>2)', 'Case': 'All values within 10 nM',
     'Condition': 'All pairwise |Δvalue| ≤ 10 nM after PubMed filter',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n>2)', 'Case': 'Values disagree by >10 nM',
     'Condition': 'At least one pair of records differs by >10 nM',
     'Action': 'Keep the value closest to the median'},

    {'Step': 'Post-filter', 'Case': 'Non-canonical amino acids',
     'Condition': 'Peptide sequence contains the "+" character',
     'Action': 'Exclude from structure generation'},
]

S1 = pd.DataFrame(s1_rows)
S1

,Step,Case,Condition,Action
0,Pre-filter,Missing quantitative measurement or assay units,`Quantitative_Measurement` or `Assay_Units` is NaN,Drop record
1,Pre-filter,Variant Kd assay labels,"`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}",Normalize to dissociation constant (KD) prior to filtering
2,Pre-filter,Non-binding-affinity assay,"`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}",Drop record
3,Dedup (n=1),Single record,—,Keep as-is
4,Dedup (n=2),PubMed provenance asymmetric,"One record has a PubMed ID, the other does not",Drop the record without PubMed ID
5,Dedup (n=2),"Both lack PubMed, values agree",Both records lack PubMed ID AND |Δvalue| ≤ 10 nM,Keep one representative (the first)
6,Dedup (n=2),"Both lack PubMed, values disagree",Both records lack PubMed ID AND |Δvalue| > 10 nM,Drop both records
7,Dedup (n=2),"Both have PubMed, values agree",Both records have PubMed ID AND |Δvalue| < 10 nM,Keep one representative
8,Dedup (n=2),"Both have PubMed, values disagree",Both records have PubMed ID AND |Δvalue| ≥ 10 nM,Flag for manual review against source publications
9,Dedup (n>2),Mixed PubMed status,"Some records have PubMed ID, others do not","Drop records without PubMed ID first, then re-evaluate"


## Table S2 — Per-allele Spearman correlations

Per-allele Spearman ρ between `rosetta_best_score` and log₁₀(measurement), computed separately for IC50 and Kd. 
Censored values at assay detection limits are excluded (IC50: 20,000 / 50,000 / 70,000 nM; Kd: 5,000 / 10,000 / 20,000 nM). 
Only alleles with n ≥ 10 quantitative measurements are reported.

In [22]:
df = pd.read_csv(METADATA_CSV)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nUnique measurement_type values:', df['measurement_type'].unique())

Shape: (49488, 20)
Columns: ['allele_iedb', 'allele', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'assay_pdb_id', 'flagged', 'has_structures', 'num_pdbs', 'rosetta_best_score', 'rosetta_mean_score', 'pdb_dir']

Unique measurement_type values: ['IC50' 'Kd']


In [23]:
# Robust filter: case-insensitive substring match
type_lower = df['measurement_type'].astype(str).str.lower()
is_ic50 = type_lower.str.contains('ic50', na=False)
is_kd   = type_lower.str.contains(r'\bkd\b', na=False, regex=True) | type_lower.str.fullmatch('kd', na=False)

print(f'IC50 rows: {is_ic50.sum():,}    Kd rows: {is_kd.sum():,}')

ic50 = df[is_ic50].copy()
kd   = df[is_kd].copy()

IC50 rows: 13,629    Kd rows: 35,859


In [24]:
MIN_N = 10
IC50_CENSORED = {20000, 50000, 70000}
KD_CENSORED   = {5000, 10000, 20000}

def per_allele_spearman(df_subset, censored, measurement_label):
    df_q = df_subset[~df_subset['measurement_value'].isin(censored)].copy()
    df_q['log_value'] = np.log10(df_q['measurement_value'])
    rows = []
    for allele, grp in df_q.groupby('allele'):
        if len(grp) < MIN_N:
            continue
        rho, p = spearmanr(grp['rosetta_best_score'], grp['log_value'])
        rows.append({
            'Allele': allele,
            'Measurement': measurement_label,
            'n': len(grp),
            'Spearman ρ': round(rho, 3),
            'p-value': p,
            'Significant (p<0.05)': 'yes' if p < 0.05 else 'no',
        })
    cols = ['Allele', 'Measurement', 'n', 'Spearman ρ', 'p-value', 'Significant (p<0.05)']
    return pd.DataFrame(rows, columns=cols)

S2_ic50 = per_allele_spearman(ic50, IC50_CENSORED, 'IC50')
S2_kd   = per_allele_spearman(kd,   KD_CENSORED,   'Kd')

print(f'S2_ic50: {len(S2_ic50)} alleles | S2_kd: {len(S2_kd)} alleles')

S2_ic50: 28 alleles | S2_kd: 25 alleles


In [25]:
# Sanity check vs. manuscript stats (run BEFORE formatting p-values as strings)
for label, sub in [('IC50', S2_ic50), ('Kd', S2_kd)]:
    if len(sub) == 0:
        print(f'{label}: no alleles met n ≥ {MIN_N} threshold')
        continue
    n_sig = (sub['p-value'] < 0.05).sum()
    print(f'{label:>4}: n_alleles={len(sub):>3}, '
          f'median ρ={sub["Spearman ρ"].median():+.3f}, '
          f'IQR [{sub["Spearman ρ"].quantile(0.25):+.2f}, {sub["Spearman ρ"].quantile(0.75):+.2f}], '
          f'{n_sig} significant at p<0.05')

IC50: n_alleles= 28, median ρ=+0.091, IQR [-0.00, +0.20], 9 significant at p<0.05
  Kd: n_alleles= 25, median ρ=+0.130, IQR [+0.06, +0.17], 14 significant at p<0.05


In [26]:
# Final S2 with formatted p-values for display
S2 = pd.concat([S2_ic50, S2_kd], ignore_index=True)
S2['p-value'] = S2['p-value'].apply(lambda x: '< 0.001' if x < 0.001 else f'{x:.3f}')
S2 = S2.sort_values(['Measurement', 'Allele']).reset_index(drop=True)
S2

,Allele,Measurement,n,Spearman ρ,p-value,Significant (p<0.05)
0,A*01:01,IC50,253,0.236,< 0.001,yes
1,A*02:01,IC50,5807,0.084,< 0.001,yes
2,A*03:01,IC50,529,-0.003,0.950,no
3,A*11:01,IC50,584,0.080,0.053,no
4,A*23:01,IC50,29,0.339,0.072,no
5,A*24:02,IC50,748,0.067,0.066,no
6,A*26:01,IC50,17,0.064,0.808,no
7,A*29:02,IC50,48,-0.087,0.554,no
8,A*30:01,IC50,10,0.139,0.701,no
9,A*30:02,IC50,31,0.395,0.028,yes


## Table S3 — Structural validation pairs

All peptide-HLA pairs used in the structural validation analysis. 
Pairs without a non-self threading template (B\*40:02-REFSKEPEL, A\*68:01-AIFQSSMTK) appear here with NaN template fields.

In [27]:
rmsd_df = pd.read_csv(RMSD_CSV)
print('Shape:', rmsd_df.shape)
print('Columns:', list(rmsd_df.columns))

Shape: (52, 33)
Columns: ['allele_iedb', 'allele', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'assay_pdb_id', 'flagged', 'has_structures', 'num_pdbs', 'rosetta_best_score', 'rosetta_mean_score', 'pdb_dir', 'matched_pdb_id', 'mhc_chain_id', 'peptide_chain_id', 'resolution_angstrom', 'rmsd_best_score', 'rmsd_top5_mean', 'rmsd_min_of_25', 'best_score', 'n_decoys', 'mhc_alignment_rmsd', 'template_pdb', 'template_peptide', 'template_identity']


In [28]:
rename_map = {
    'allele':               'Allele',
    'peptide':               'Peptide',
    'peptide_length':        'Length',
    'matched_pdb_id':        'Experimental PDB',
    'template_pdb':          'Template',
    'template_identity':     'Template identity (%)',
    'rmsd_best_score':       'RMSD best-by-score (Å)',
    'rmsd_top5_mean':        'RMSD top-5 mean (Å)',
    'rmsd_min_of_25':        'RMSD best-overall (Å)',
}
keep_cols = [c for c in rename_map if c in rmsd_df.columns]
S3 = rmsd_df[keep_cols].rename(columns=rename_map).copy()

# Template identity to %: if stored as fraction ≤ 1, multiply by 100
if 'Template identity (%)' in S3.columns and S3['Template identity (%)'].dropna().max() <= 1.5:
    S3['Template identity (%)'] = S3['Template identity (%)'] * 100

# Round numeric columns
for c in ['RMSD best-by-score (Å)', 'RMSD top-5 mean (Å)', 'RMSD best-overall (Å)']:
    if c in S3.columns: S3[c] = S3[c].round(2)
if 'Template identity (%)' in S3.columns:
    S3['Template identity (%)'] = S3['Template identity (%)'].round(1)

# Sort by template identity descending (no-template pairs sink to bottom)
if 'Template identity (%)' in S3.columns:
    S3 = S3.sort_values('Template identity (%)', ascending=False, na_position='last').reset_index(drop=True)

print(f'{len(S3)} rows')
S3

52 rows


,Allele,Peptide,Length,Experimental PDB,Template,Template identity (%),RMSD best-by-score (Å),RMSD top-5 mean (Å),RMSD best-overall (Å)
0,A*02:01,ALWGPDPAAA,10,3UTQ,5C0D,90.0,0.79,0.73,0.58
1,B*27:05,KRWIILGLNK,10,4G9D,4G8I,90.0,0.94,1.14,0.94
2,A*02:01,ITDQVPFSV,9,1TVB,6VMC,88.9,0.75,0.85,0.75
3,A*02:01,CINGVCWTV,9,3MRG,3MRJ,88.9,1.11,1.05,0.76
4,A*02:01,NLVPMVATV,9,6Q3K,3GSR,88.9,1.18,1.21,1.03
5,A*02:01,VLHDDLLEA,9,3D25,3FT4,88.9,0.89,0.99,0.89
6,A*02:01,SLYNTVATL,9,2V2W,5NMH,88.9,0.97,1.00,0.91
7,A*02:01,AAGIGILTV,9,3QFD,2GTZ,88.9,1.40,1.35,0.92
8,A*02:01,YLQPRTFLL,9,7P3D,7P3E,88.9,1.27,1.32,1.23
9,A*02:01,GILGFVFTL,9,1OGA,5HHQ,88.9,1.47,1.30,0.81


## Table S4 — Per-allele dataset composition

Per-allele peptide counts, IC50/Kd splits, and peptide-length range. Supports the imbalance caveat in Usage Notes.

In [29]:
comp_rows = []
for allele, grp in df.groupby('allele'):
    n_pep = grp['peptide'].nunique()
    n_pairs = grp[['allele', 'peptide']].drop_duplicates().shape[0]
    n_ic50 = is_ic50.loc[grp.index].sum()
    n_kd   = is_kd.loc[grp.index].sum()
    lens = grp['peptide'].dropna().str.len()
    comp_rows.append({
        'Allele': allele,
        'Unique peptides':   int(n_pep),
        'Peptide-HLA pairs': int(n_pairs),
        'IC50 measurements': int(n_ic50),
        'Kd measurements':   int(n_kd),
        'Length range':      f'{lens.min()}-{lens.max()}' if len(lens) else '—',
    })

S4 = pd.DataFrame(comp_rows).sort_values('Peptide-HLA pairs', ascending=False).reset_index(drop=True)
S4['% of dataset'] = (S4['Peptide-HLA pairs'] / S4['Peptide-HLA pairs'].sum() * 100).round(2)
S4

,Allele,Unique peptides,Peptide-HLA pairs,IC50 measurements,Kd measurements,Length range,% of dataset
0,A*02:01,9102,9102,6195,3039,7-15,18.47
1,A*68:02,3457,3457,2451,1006,8-11,7.02
2,A*03:01,3097,3097,591,2516,8-12,6.29
3,B*15:01,2802,2802,283,2535,8-14,5.69
4,A*11:01,2508,2508,667,1846,8-11,5.09
5,B*07:02,2441,2441,561,1904,8-12,4.95
6,A*01:01,2313,2313,299,2031,8-12,4.69
7,B*58:01,2078,2078,39,2039,8-11,4.22
8,A*26:01,1994,1994,29,1965,9-11,4.05
9,B*57:01,1954,1954,90,1864,9-14,3.97


## Save all tables to one .xlsx

One sheet per table — open in Excel, copy individual tables into Word.

In [30]:
out_xlsx = OUT_DIR / 'PepBind3D_supplemental_tables.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as xw:
    S1.to_excel(xw, sheet_name='S1_dedup_logic',         index=False)
    S2.to_excel(xw, sheet_name='S2_per_allele_spearman', index=False)
    S3.to_excel(xw, sheet_name='S3_validation_pairs',    index=False)
    S4.to_excel(xw, sheet_name='S4_allele_composition',  index=False)

print(f'Wrote: {out_xlsx}')

Wrote: /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/supplemental_tables/PepBind3D_supplemental_tables.xlsx
